In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D4 — Eurostat — LFS Metadata Excel
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import re


import pandas as pd
import openpyxl

DOCUMENT_ID = "D4"
DOCUMENT_NAME = "Eurostat — LFS Metadata Excel — lfsa_esms"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "Complete XLSX workbook-to-Markdown structural conversion "
    "with explicit source-row and target-scope hierarchy markers"
)

EXPECTED_SOURCE_FORMAT = ".xlsx"
EXPECTED_SOURCE_SHA256 = "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9"

EXPECTED_SHEETS = [
    "Metadata",
    "Parameters",
    "Annexes"
]

SOURCE_SHEET = "Metadata"
EXPECTED_METADATA_ROW_COUNT = 86

EXPECTED_SCOPE_ROWS = (
    list(range(2, 10))
    + list(range(12, 87))
)

EXPECTED_RECORD_COUNT = 83
EXPECTED_HEADER_RECORD_COUNT = 8
EXPECTED_CONCEPT_RECORD_COUNT = 75

EXPECTED_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

ALLOWED_PUBLICATION_FLAGS = {
    "YES",
    "NO",
    None
}

OUTPUT_DIR = Path("outputs_D4_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

uploaded = files.upload()

xlsx_files = [
    Path(name)
    for name in uploaded.keys()
    if name.lower().endswith(".xlsx")
]

if len(xlsx_files) != 1:
    raise ValueError("Upload exactly one D4 XLSX workbook.")

WORKBOOK_PATH = xlsx_files[0]

print("Uploaded workbook:", WORKBOOK_PATH.name)


In [ ]:
# ============================================================
# 2. Verify source format and frozen SHA-256 identity
# ============================================================

if WORKBOOK_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FORMAT}; "
        f"received {WORKBOOK_PATH.suffix}"
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

WORKBOOK_SHA256 = sha256_file(WORKBOOK_PATH)
SOURCE_HASH_MATCH = (
    WORKBOOK_SHA256 == EXPECTED_SOURCE_SHA256
)

print("Observed SHA-256:", WORKBOOK_SHA256)
print("Matches frozen D4 source:", SOURCE_HASH_MATCH)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded workbook does not match the frozen D4 source identity."
    )


In [ ]:
# ============================================================
# 3. Inspect complete workbook structure
# ============================================================

excel_file = pd.ExcelFile(WORKBOOK_PATH)
sheet_names = excel_file.sheet_names

if sheet_names != EXPECTED_SHEETS:
    raise ValueError(
        f"Unexpected worksheet order/content. "
        f"Expected {EXPECTED_SHEETS}; observed {sheet_names}"
    )

workbook = openpyxl.load_workbook(
    WORKBOOK_PATH,
    data_only=False
)

workbook_structure = []

for sheet_name in sheet_names:
    ws = workbook[sheet_name]
    workbook_structure.append({
        "sheet_name": sheet_name,
        "max_row": int(ws.max_row),
        "max_column": int(ws.max_column)
    })

print(json.dumps(workbook_structure, indent=2))


In [ ]:
# ============================================================
# 4. Verify the fixed Stage 1 Metadata scope
# ============================================================

metadata_ws = workbook[SOURCE_SHEET]

metadata_row_count_valid = (
    metadata_ws.max_row >= EXPECTED_METADATA_ROW_COUNT
)

structural_label_checks = {
    "row_1_header": metadata_ws.cell(1, 1).value == "Header",
    "row_10_concepts": metadata_ws.cell(10, 1).value == "Concepts",
    "row_11_concept_name":
        metadata_ws.cell(11, 1).value == "Concept name",
    "row_11_concept_value":
        metadata_ws.cell(11, 2).value == "Concept value",
    "row_11_restricted":
        metadata_ws.cell(11, 3).value == "Restricted from publication"
}

scope_rows_present = all(
    metadata_ws.cell(row, 1).value is not None
    for row in EXPECTED_SCOPE_ROWS
)

scope_check = {
    "metadata_row_count_at_least_86": metadata_row_count_valid,
    "structural_label_checks": structural_label_checks,
    "structural_labels_valid":
        all(structural_label_checks.values()),
    "expected_scope_rows": "2–9 and 12–86",
    "expected_scope_row_count": len(EXPECTED_SCOPE_ROWS),
    "scope_rows_present": scope_rows_present
}

print(json.dumps(scope_check, indent=2, ensure_ascii=False))

if not all([
    metadata_row_count_valid,
    all(structural_label_checks.values()),
    scope_rows_present
]):
    raise ValueError("The fixed D4 Stage 1 scope could not be verified.")


## D4 structural-conversion rule

The complete workbook is retained in Branch B because Branch A exposes the complete workbook to the model.

The conversion therefore creates:

1. a complete Markdown section for every worksheet;
2. explicit source-row boundaries;
3. explicit source-column positions;
4. an additional structural interpretation for the **target Metadata rows** that makes the existing section hierarchy visible.

The source cells remain unchanged. In particular:

- HTML-like text is not cleaned or rendered;
- hyperlinks are not followed;
- HTML entities are not decoded;
- empty cells are not semantically filled;
- publication flags are not harmonised;
- labels are not corrected;
- no reference values are used to construct the representation.

In [ ]:
# ============================================================
# 5. Build deterministic target-scope structural records
# ============================================================

TOP_LEVEL_RE = re.compile(r"^(\d+)\.\s+")
SUBSECTION_RE = re.compile(r"^(\d+)\.(\d+)\.\s+")

def preserve_cell(value):
    if value is None:
        return None
    if isinstance(value, str) and value == "":
        return None
    return value

target_records = []
current_section = None

# Header records: Metadata rows 2–9
for row in range(2, 10):
    target_records.append({
        "Source Row": row,
        "Section": "Header",
        "Concept Name": preserve_cell(
            metadata_ws.cell(row, 1).value
        ),
        "Concept Value": preserve_cell(
            metadata_ws.cell(row, 2).value
        ),
        "Publication Restricted": preserve_cell(
            metadata_ws.cell(row, 3).value
        )
    })

# Concept records: Metadata rows 12–86
for row in range(12, 87):
    concept_name = preserve_cell(
        metadata_ws.cell(row, 1).value
    )

    if isinstance(concept_name, str):
        if TOP_LEVEL_RE.match(concept_name):
            current_section = concept_name

    target_records.append({
        "Source Row": row,
        "Section": current_section,
        "Concept Name": concept_name,
        "Concept Value": preserve_cell(
            metadata_ws.cell(row, 2).value
        ),
        "Publication Restricted": preserve_cell(
            metadata_ws.cell(row, 3).value
        )
    })

target_df = pd.DataFrame(target_records)

print("Target structural records:", len(target_df))
print("Header records:",
      int((target_df["Section"] == "Header").sum()))
print("Concept records:",
      int((target_df["Section"] != "Header").sum()))


In [ ]:
# ============================================================
# 6. Verify target-record construction
# ============================================================

target_record_count_valid = (
    len(target_df) == EXPECTED_RECORD_COUNT
)

header_record_count = int(
    (target_df["Section"] == "Header").sum()
)

concept_record_count = (
    len(target_df) - header_record_count
)

header_count_valid = (
    header_record_count == EXPECTED_HEADER_RECORD_COUNT
)

concept_count_valid = (
    concept_record_count == EXPECTED_CONCEPT_RECORD_COUNT
)

missing_concept_names = int(
    target_df["Concept Name"].isna().sum()
)

missing_sections = int(
    target_df["Section"].isna().sum()
)

target_TECHNICAL_DIAGNOSTICS = {
    "target_record_count_valid": target_record_count_valid,
    "observed_target_records": int(len(target_df)),
    "header_count_valid": header_count_valid,
    "observed_header_records": header_record_count,
    "concept_count_valid": concept_count_valid,
    "observed_concept_records": concept_record_count,
    "missing_concept_names": missing_concept_names,
    "missing_sections": missing_sections
}

print(json.dumps(target_TECHNICAL_DIAGNOSTICS, indent=2))

if not all([
    target_record_count_valid,
    header_count_valid,
    concept_count_valid,
    missing_concept_names == 0,
    missing_sections == 0
]):
    raise ValueError(
        "Target structural-record construction failed."
    )


In [ ]:
# ============================================================
# 7. Markdown-preservation helpers
# ============================================================

def markdown_inline(value):
    if value is None:
        return "`null`"

    text = str(value)

    # Backticks are escaped only for Markdown syntax preservation.
    text = text.replace("`", "\\`")
    return f"`{text}`"

def fenced_source_value(value):
    if value is None:
        return "```text\nnull\n```"

    # Use a fenced block so HTML-like source strings remain literal.
    return "```text\n" + str(value) + "\n```"


In [ ]:
# ============================================================
# 8. Convert the COMPLETE workbook to Markdown
# ============================================================

markdown_lines = [
    "# D4 — Eurostat LFS Metadata",
    "",
    "> Structural conversion of the complete original XLSX workbook.",
    "> The fixed extraction task remains restricted to Metadata rows 2–9 and 12–86.",
    ""
]

complete_source_cell_count = 0

for sheet_name in sheet_names:
    ws = workbook[sheet_name]

    markdown_lines.append(
        f"## Worksheet: {sheet_name}"
    )
    markdown_lines.append("")

    for row in range(1, ws.max_row + 1):
        values = [
            ws.cell(row, col).value
            for col in range(1, ws.max_column + 1)
        ]

        if all(v is None for v in values):
            continue

        markdown_lines.append(
            f"### Source Row {row}"
        )
        markdown_lines.append("")

        for col, value in enumerate(values, start=1):
            if value is None:
                rendered = "`null`"
            else:
                complete_source_cell_count += 1
                rendered = fenced_source_value(value)

            markdown_lines.append(
                f"**Column {col}**"
            )
            markdown_lines.append("")
            markdown_lines.append(rendered)
            markdown_lines.append("")

    # Add an explicit structural interpretation only for the
    # fixed target rows of the Metadata worksheet.
    if sheet_name == SOURCE_SHEET:
        markdown_lines.append(
            "## Target-scope structural interpretation"
        )
        markdown_lines.append("")

        for _, record in target_df.iterrows():
            markdown_lines.append(
                f"### Target Record — Source Row {int(record['Source Row'])}"
            )
            markdown_lines.append("")
            markdown_lines.append(
                f"- Section: {markdown_inline(record['Section'])}"
            )
            markdown_lines.append(
                f"- Concept Name: {markdown_inline(record['Concept Name'])}"
            )
            markdown_lines.append(
                "- Concept Value:"
            )
            markdown_lines.append(
                fenced_source_value(record["Concept Value"])
            )
            markdown_lines.append(
                f"- Publication Restricted: "
                f"{markdown_inline(record['Publication Restricted'])}"
            )
            markdown_lines.append("")

MARKDOWN_TEXT = "\n".join(markdown_lines).rstrip() + "\n"

MARKDOWN_PATH = (
    OUTPUT_DIR / "D4_branch_B_structural_markdown.md"
)

MARKDOWN_PATH.write_text(
    MARKDOWN_TEXT,
    encoding="utf-8"
)

MARKDOWN_SHA256 = sha256_file(MARKDOWN_PATH)

print("Markdown saved:", MARKDOWN_PATH)
print("Markdown SHA-256:", MARKDOWN_SHA256)
print("Characters:", len(MARKDOWN_TEXT))


In [ ]:
# ============================================================
# 9. Verify complete-workbook content preservation
# ============================================================

missing_source_cells = []

for sheet_name in sheet_names:
    ws = workbook[sheet_name]

    for row in range(1, ws.max_row + 1):
        for col in range(1, ws.max_column + 1):
            value = ws.cell(row, col).value

            if value is None:
                continue

            if str(value) not in MARKDOWN_TEXT:
                missing_source_cells.append({
                    "sheet": sheet_name,
                    "row": row,
                    "column": col,
                    "value_preview": str(value)[:200]
                })

# Verify target source values independently.
missing_target_values = []

for _, record in target_df.iterrows():
    for field in [
        "Concept Name",
        "Concept Value",
        "Publication Restricted"
    ]:
        value = record[field]
        if value is None:
            continue

        if str(value) not in MARKDOWN_TEXT:
            missing_target_values.append({
                "source_row": int(record["Source Row"]),
                "field": field,
                "value_preview": str(value)[:200]
            })

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": WORKBOOK_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "observed_sheets": sheet_names,
    "expected_sheets": EXPECTED_SHEETS,
    "conversion_method": CONVERSION_METHOD,
    "complete_workbook_converted": True,
    "worksheet_filtering_applied": False,
    "out_of_scope_rows_removed": False,
    "out_of_scope_worksheets_removed": False,
    "fixed_target_scope": "Metadata rows 2–9 and 12–86",
    "expected_target_record_count": EXPECTED_RECORD_COUNT,
    "target_record_count": int(len(target_df)),
    "target_record_count_valid": target_record_count_valid,
    "header_count_valid": header_count_valid,
    "concept_count_valid": concept_count_valid,
    "source_cell_count_checked": int(complete_source_cell_count),
    "missing_source_cell_count": len(missing_source_cells),
    "missing_source_cells": missing_source_cells,
    "missing_target_value_count": len(missing_target_values),
    "missing_target_values": missing_target_values,
    "html_cleaning_applied": False,
    "html_rendering_applied": False,
    "html_entity_decoding_applied": False,
    "hyperlinks_followed": False,
    "label_standardisation_applied": False,
    "semantic_harmonisation_applied": False,
    "value_modification_applied": False,
    "manual_correction_applied": False,
    "manual_reconstruction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed": all([
        SOURCE_HASH_MATCH,
        sheet_names == EXPECTED_SHEETS,
        target_record_count_valid,
        header_count_valid,
        concept_count_valid,
        len(missing_source_cells) == 0,
        len(missing_target_values) == 0
    ])
}

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D4_branch_B_conversion_integrity.json"
)

with open(CONVERSION_INTEGRITY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        CONVERSION_INTEGRITY,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    CONVERSION_INTEGRITY,
    indent=2,
    ensure_ascii=False
))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError("D4 Branch B conversion integrity failed.")


In [ ]:
# ============================================================
# 10. Preserve conversion audit and representation metadata
# ============================================================

TARGET_RECORDS_PATH = (
    OUTPUT_DIR / "D4_branch_B_target_structural_records.csv"
)
target_df.to_csv(
    TARGET_RECORDS_PATH,
    index=False
)

REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete XLSX workbook converted to structured Markdown",
    "source_file": WORKBOOK_PATH.name,
    "source_format": WORKBOOK_PATH.suffix.lower(),
    "source_sha256": WORKBOOK_SHA256,
    "representation_file": MARKDOWN_PATH.name,
    "representation_sha256": MARKDOWN_SHA256,
    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        "Structural Markdown",
    "all_source_worksheets_retained": True,
    "all_nonempty_source_rows_retained": True,
    "fixed_extraction_scope":
        "Metadata rows 2–9 and 12–86",
    "target_scope_structural_interpretation_added": True,
    "source_row_metadata_added": True,
    "source_row_metadata_requested_from_model": False,
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "html_cleaning_applied": False,
    "semantic_harmonisation_applied": False,
    "worksheet_filtering_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D4_branch_B_representation.json"
)

with open(REPRESENTATION_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        REPRESENTATION_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    REPRESENTATION_METADATA,
    indent=2,
    ensure_ascii=False
))


## Frozen extraction controls

The extraction task below is the Branch B counterpart of the final Branch A task.

The substantive scope, record count, output fields, source-preservation requirements and exclusions remain unchanged. Only representation-dependent wording changes from **original Excel workbook** to **structurally converted Markdown**.

The Stage 1 reference dataset is never supplied to the model.

In [ ]:
# ============================================================
# 11. Define the fixed output schema
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "records": [
        {
            "Section": None,
            "Concept Name": None,
            "Concept Value": None,
            "Publication Restricted": None
        }
    ]
}

print(json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 12. Operationalise the fixed Stage 1 extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every metadata record represented within the defined scope of
the Metadata worksheet in the attached structurally converted Markdown
document.

For every included record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Treat the attached structurally converted Markdown document as the
  only source of information.
- Use only the content corresponding to the Metadata worksheet.
- Include the workbook-header metadata records corresponding to source
  rows 2–9.
- Include every metadata concept record corresponding to source rows
  12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude source row 1, labelled "Header".
- Exclude source row 10, labelled "Concepts".
- Exclude source row 11, containing the source column headings.
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.

Extraction rules:

- Preserve Concept Name exactly as represented in the source.
- Preserve Concept Value exactly as represented, including HTML-like
  tags, hyperlinks, attributes, entities, punctuation and internal line
  breaks.
- Do not render, simplify, remove or rewrite HTML-like content.
- Preserve YES and NO publication-restriction flags exactly.
- Use null when a Concept Value or Publication Restricted cell is
  genuinely empty.
- Assign workbook-header records from source rows 2–9 to Section
  "Header".
- For numbered metadata concepts, use the complete top-level numbered
  section heading as Section, for example "1. Contact".
- Do not follow hyperlinks.
- Do not infer missing values.
- Do not calculate, summarise, paraphrase, translate, harmonise or
  correct source content.
- Do not use external knowledge.
- Return one record for every included source row.
- Verify that only the fixed Metadata worksheet scope has been
  processed.
- Verify that every record within that scope has been processed.
- Verify that empty source cells are represented as null.
- Verify that HTML-like source content remains unchanged.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.
""".strip()

print(EXTRACTION_TASK)


In [ ]:
# ============================================================
# 13. Construct and preserve the Branch B prompt
# ============================================================

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The complete structurally converted representation of the original
XLSX workbook is attached.

The extraction scope remains restricted to Metadata source rows 2–9
and 12–86. Other workbook content is present only to preserve the
same overall source exposure as Branch A and must not be extracted.

Return only the JSON object.
""".strip()

PROMPT_PATH = (
    OUTPUT_DIR / "D4_branch_B_prompt.txt"
)

PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 14. Create Branch B experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": WORKBOOK_PATH.name,
    "source_format": WORKBOOK_PATH.suffix.lower(),
    "source_sha256": WORKBOOK_SHA256,
    "llm_input_representation":
        "Structural Markdown",

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": MARKDOWN_PATH.name,
    "representation_sha256": MARKDOWN_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "complete_workbook_retained": True,
    "worksheet_filtering_applied": False,
    "out_of_scope_content_retained": True,
    "normalisation_applied": False,
    "html_cleaning_applied": False,
    "html_rendering_applied": False,
    "hyperlinks_followed": False,
    "semantic_harmonisation_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "content_validation_performed": False,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "expected_header_record_count":
        EXPECTED_HEADER_RECORD_COUNT,
    "expected_concept_record_count":
        EXPECTED_CONCEPT_RECORD_COUNT,
    "expected_fields": EXPECTED_FIELDS,
    "fixed_extraction_scope":
        "Metadata rows 2–9 and 12–86",
    "conversion_integrity_file":
        CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format": "JSON",
    "execution_environment":
        "Independent ChatGPT conversation",
    "notes": (
        "Only the representation pathway changes relative to Branch A. "
        "The complete workbook remains present in the Branch B structural "
        "Markdown, while the fixed extraction task remains restricted to "
        "Metadata rows 2–9 and 12–86. HTML-like values are preserved "
        "without cleaning, rendering or semantic rewriting. No Stage 1 "
        "reference values are supplied to the model. Content-level "
        "validation is performed separately in Validation B — D4."
    )
}

METADATA_PATH = (
    OUTPUT_DIR / "D4_branch_B_experiment_metadata.json"
)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        EXPERIMENT_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    EXPERIMENT_METADATA,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 15. Download model-input artefacts
# ============================================================

for path in [
    MARKDOWN_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    REPRESENTATION_METADATA_PATH,
    CONVERSION_INTEGRITY_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D4_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D4_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original XLSX or Stage 1 reference values.\n"
    "5. Do not manually correct, regenerate, or repair the response.\n"
    "6. Save the complete response exactly as returned."
)


In [ ]:
# ============================================================
# 16. Upload the untouched Branch B model response
# ============================================================

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete "
        "D4 Branch B model response."
    )

UPLOADED_RAW_OUTPUT = Path(
    next(iter(uploaded_output))
)

print("Uploaded raw response:",
      UPLOADED_RAW_OUTPUT)


In [ ]:
# ============================================================
# 17. Preserve the raw response before parsing
# ============================================================

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D4_branch_B_raw_response.txt"
)

raw_response_text = (
    UPLOADED_RAW_OUTPUT.read_text(
        encoding="utf-8"
    )
)

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = (
    sha256_file(RAW_RESPONSE_PATH)
)

print("Raw response preserved.")
print("Raw response SHA-256:",
      RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 18. Parse the raw response without modifying it
# ============================================================

valid_json = False
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(
        raw_response_text
    )
    valid_json = True
except json.JSONDecodeError as exc:
    json_error = str(exc)

print("Valid JSON:", valid_json)

if json_error:
    print("JSON parsing error:", json_error)


In [ ]:
# ============================================================
# 19. Validate top-level structure
# ============================================================

top_level_object_valid = (
    valid_json
    and isinstance(raw_extraction, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in raw_extraction
)

document_id_correct = (
    document_id_present
    and raw_extraction.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in raw_extraction
)

branch_correct = (
    branch_present
    and raw_extraction.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in raw_extraction
)

records_is_list = (
    records_present
    and isinstance(
        raw_extraction.get("records"),
        list
    )
)

records = (
    raw_extraction["records"]
    if records_is_list
    else []
)

print({
    "top_level_object_valid":
        top_level_object_valid,
    "document_id_correct":
        document_id_correct,
    "branch_correct":
        branch_correct,
    "records_is_list":
        records_is_list
})


In [ ]:
# ============================================================
# 20. Validate record schemas and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
publication_flag_issues = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(
        expected_fields - actual_fields
    )
    extra_fields = sorted(
        actual_fields - expected_fields
    )

    if missing_fields or extra_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields
        })

    for field in EXPECTED_FIELDS:
        value = record.get(field)

        if (
            value is not None
            and not isinstance(value, str)
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type":
                    type(value).__name__
            })

    flag = record.get(
        "Publication Restricted"
    )

    if flag not in ALLOWED_PUBLICATION_FLAGS:
        publication_flag_issues.append({
            "record_index": record_index,
            "observed_flag": flag
        })

print("Record structure issues:",
      len(record_structure_issues))
print("Field type issues:",
      len(field_type_issues))
print("Publication flag issues:",
      len(publication_flag_issues))


In [ ]:
# ============================================================
# 21. Scope and duplicate diagnostics
# ============================================================

record_count = len(records)

record_count_valid = (
    record_count == EXPECTED_RECORD_COUNT
)

header_record_count = sum(
    1
    for record in records
    if (
        isinstance(record, dict)
        and record.get("Section") == "Header"
    )
)

concept_record_count = (
    record_count - header_record_count
)

header_count_valid = (
    header_record_count
    == EXPECTED_HEADER_RECORD_COUNT
)

concept_count_valid = (
    concept_record_count
    == EXPECTED_CONCEPT_RECORD_COUNT
)

record_keys = [
    (
        record.get("Section"),
        record.get("Concept Name")
    )
    for record in records
    if isinstance(record, dict)
]

duplicate_record_key_count = (
    len(record_keys)
    - len(set(record_keys))
)

missing_values_by_field = {
    field: sum(
        1
        for record in records
        if (
            not isinstance(record, dict)
            or field not in record
            or record.get(field) is None
        )
    )
    for field in EXPECTED_FIELDS
}

print("Expected records:", EXPECTED_RECORD_COUNT)
print("Observed records:", record_count)
print("Header records:", header_record_count)
print("Concept records:", concept_record_count)
print("Duplicate keys:", duplicate_record_key_count)
print("Missing values:", missing_values_by_field)


In [ ]:
# ============================================================
# 22. Create structural/schema diagnostics
# ============================================================

record_schema_valid = (
    len(record_structure_issues) == 0
)

field_types_valid = (
    len(field_type_issues) == 0
)

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid,
    field_types_valid
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,
    "branch":
        BRANCH,

    "valid_json":
        bool(valid_json),
    "json_error":
        json_error,

    "top_level_object_valid":
        bool(top_level_object_valid),
    "document_id_present":
        bool(document_id_present),
    "document_id_correct":
        bool(document_id_correct),
    "branch_present":
        bool(branch_present),
    "branch_correct":
        bool(branch_correct),
    "records_present":
        bool(records_present),
    "records_is_list":
        bool(records_is_list),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        int(record_count),
    "record_count_valid":
        bool(record_count_valid),

    "expected_header_record_count":
        EXPECTED_HEADER_RECORD_COUNT,
    "observed_header_record_count":
        int(header_record_count),
    "header_count_valid":
        bool(header_count_valid),

    "expected_concept_record_count":
        EXPECTED_CONCEPT_RECORD_COUNT,
    "observed_concept_record_count":
        int(concept_record_count),
    "concept_count_valid":
        bool(concept_count_valid),

    "records_with_structure_issues":
        len(record_structure_issues),
    "record_structure_issues":
        record_structure_issues,

    "field_type_issue_count":
        len(field_type_issues),
    "field_type_issues":
        field_type_issues,

    "publication_flag_issue_count":
        len(publication_flag_issues),
    "publication_flag_issues":
        publication_flag_issues,

    "duplicate_record_key_count":
        int(duplicate_record_key_count),

    "missing_values_by_field":
        missing_values_by_field,

    "record_schema_valid":
        bool(record_schema_valid),
    "field_types_valid":
        bool(field_types_valid),

    "scope_complete":
        bool(
            record_count_valid
            and header_count_valid
            and concept_count_valid
        ),

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D4_branch_B_technical_diagnostics.json"
)

with open(TECHNICAL_DIAGNOSTICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        TECHNICAL_DIAGNOSTICS,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    TECHNICAL_DIAGNOSTICS,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 23. Preserve parsed extraction only when JSON is valid
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D4_branch_B_parsed_extraction.json"
)

if valid_json:
    with open(PARSED_EXTRACTION_PATH, "w", encoding="utf-8") as f:
        json.dump(
            raw_extraction,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Parsed extraction saved:",
          PARSED_EXTRACTION_PATH)
else:
    print(
        "No parsed extraction created because "
        "the preserved raw response is invalid JSON."
    )


In [ ]:
# ============================================================
# 24. Create experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": WORKBOOK_PATH.name,
    "source_sha256": WORKBOOK_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        "Structural Markdown",
    "conversion_method":
        CONVERSION_METHOD,
    "representation_file":
        MARKDOWN_PATH.name,
    "representation_sha256":
        MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_workbook_retained": True,
    "worksheet_filtering_applied": False,
    "normalisation_applied": False,
    "html_cleaning_applied": False,
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        int(record_count),
    "valid_json":
        bool(valid_json),
    "record_count_valid":
        bool(record_count_valid),
    "structurally_evaluable":
        bool(structurally_evaluable),
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "scope_complete":
        bool(TECHNICAL_DIAGNOSTICS["scope_complete"]),
    "records_with_structure_issues":
        len(record_structure_issues),
    "field_type_issue_count":
        len(field_type_issues),
    "publication_flag_issue_count":
        len(publication_flag_issues),
    "duplicate_record_key_count":
        int(duplicate_record_key_count),
    "raw_response_preserved": True,
    "raw_response_sha256":
        RAW_RESPONSE_SHA256,
    "parsed_extraction_created":
        bool(valid_json),
    "content_validation_performed":
        False,

    "notes":
        "Content-level validation is performed separately in Validation B — D4."
}

SUMMARY_PATH = (
    OUTPUT_DIR / "D4_branch_B_experiment_summary.json"
)

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        EXPERIMENT_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    EXPERIMENT_SUMMARY,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 25. Final artefact inventory and downloads
# ============================================================

generated_outputs = [
    MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    TARGET_RECORDS_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SUMMARY_PATH
]

if valid_json:
    generated_outputs.append(
        PARSED_EXTRACTION_PATH
    )

print("Generated D4 Branch B outputs:")
for path in generated_outputs:
    print("-", path.name)

for path in generated_outputs:
    files.download(path)

print(
    "\nFor Validation B, reuse the frozen D4 validation rules: "
    "alignment identity = Section + Concept Name; "
    "formal primary correctness and Field Accuracy exclude "
    "the alignment-identity fields; HTML/content normalisation "
    "remains diagnostic only."
)